In [2]:
"""
Run this locally (not in this sandbox -- it can't reach api.deadlock-api.com).

Hits every endpoint that's plausibly relevant to hero/item analysis, pulls ONE
sample response from each, and prints:
  - the endpoint
  - every column/field name it returned
  - one example row so you can see actual values, not just names

Nothing gets merged or cleaned here -- this is purely "show me everything that
exists" so you can decide what's worth keeping.

Requires: pip install requests pandas
"""

import requests
import pandas as pd

BASE = "https://api.deadlock-api.com"
HEADERS = {"User-Agent": "python-requests (deadlock-exploration-script)"}

# path, params -- one small/cheap call per endpoint, just to see the shape
ENDPOINTS = {
    "heroes":               ("/v1/assets/heroes", {}),
    "items":                ("/v1/assets/items", {}),
    "hero-stats":           ("/v1/analytics/hero-stats", {}),
    "item-stats":           ("/v1/analytics/item-stats", {"hero_id": 1}),
    "build-item-stats":     ("/v1/analytics/build-item-stats", {"hero_id": 1}),
    "item-flow-stats":      ("/v1/analytics/item-flow-stats", {"hero_id": 1}),
    "item-permutation-stats": ("/v1/analytics/item-permutation-stats", {"hero_id": 1}),
    "ability-order-stats":  ("/v1/analytics/ability-order-stats", {"hero_id": 1}),
    "hero-ban-stats":       ("/v1/analytics/hero-ban-stats", {}),
    "hero-comb-stats":      ("/v1/analytics/hero-comb-stats", {}),
    "hero-counter-stats":   ("/v1/analytics/hero-counter-stats", {}),
    "hero-synergy-stats":   ("/v1/analytics/hero-synergy-stats", {}),
    "kill-death-stats":     ("/v1/analytics/kill-death-stats", {"hero_id": 1}),
    "player-performance-curve": ("/v1/analytics/player-performance-curve", {"hero_id": 1}),
    "badge-distribution":   ("/v1/analytics/badge-distribution", {}),
    "hero-scoreboard":      ("/v1/analytics/scoreboards/heroes", {"sort_by": "matches"}),
    "player-scoreboard":    ("/v1/analytics/scoreboards/players", {"sort_by": "matches"}),
    "hero-build-stats":     ("/v1/analytics/hero-build-stats/1", {}),
    "lane-matchup-stats":   ("/v1/analytics/lane-matchup-stats", {}),
    "lane-soul-curve":      ("/v1/analytics/lane-soul-curve", {}),
}

for label, (path, params) in ENDPOINTS.items():
    print("=" * 80)
    print(f"{label}   ({path})")
    try:
        r = requests.get(f"{BASE}{path}", params=params, headers=HEADERS, timeout=30)
        r.raise_for_status()
        data = r.json()

        # some endpoints return a list, some return a dict -- handle both
        sample = data[0] if isinstance(data, list) and len(data) > 0 else data

        if isinstance(sample, dict):
            print(f"columns: {list(sample.keys())}")
            print("example row:")
            print(sample)
        else:
            print("unexpected shape, raw response:")
            print(data)

    except requests.HTTPError as e:
        print(f"FAILED: {e}  (body: {r.text[:300]})")
    except Exception as e:
        print(f"FAILED: {e}")

        print()

heroes   (/v1/assets/heroes)
columns: ['id', 'class_name', 'name', 'description', 'player_selectable', 'disabled', 'in_development', 'needs_testing', 'assigned_players_only', 'tags', 'gun_tag', 'hideout_rich_presence', 'hero_type', 'prerelease_only', 'limited_testing', 'complexity', 'skin', 'images', 'items', 'starting_stats', 'item_slot_info', 'physics', 'colors', 'shop_stat_display', 'cost_bonuses', 'stats_display', 'hero_stats_ui', 'level_info', 'scaling_stats', 'purchase_bonuses', 'standard_level_up_upgrades', 'item_draft_bucketing']
example row:
{'id': 1, 'class_name': 'hero_inferno', 'name': 'Infernus', 'description': {'lore': 'Like most teenagers; Infernus was wild, rebellious, and impetuous.  Unlike most teenagers, Infernus was a creature from another plane and had a supernatural mastery over fire.  Needless to say:  his youth was filled with no small amount of arson, murder, and evidence disposal.  But that was then.  Now an adult, Infernus has mellowed out considerably.  He’s